## Grid Search for Analyzer Settings

In [ ]:
# Without extension
OUTPUT_NAME = 'grid_search_node2vec_2021-07-01'

## Imports

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

from IPython.display import display
import seaborn as sns
sns.set_style("whitegrid")
import matplotlib.pyplot as plt

import json
import itertools
import logging
import os
import pandas as pd

from sklearn.metrics.cluster import adjusted_mutual_info_score, v_measure_score
from sklearn.model_selection import ParameterGrid

from pysrc.papers.analysis.graph import node2vec
from pysrc.papers.analyzer import PapersAnalyzer
from pysrc.papers.config import AnalyzerSettings

from utils.analysis import get_direct_references_subgraph, align_clusterings_for_sklearn
from utils.io import reload_exported_analyzer, load_analyzer, load_clustering, get_review_pmids
from utils.metrics import pd_score, reg_v_score
from utils.preprocessing import preprocess_clustering, get_clustering_level

In [ ]:
# Configure logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s %(levelname)s: %(message)s')
ch.setFormatter(formatter)
logger.addHandler(ch)

## Analyze ground truth clustering

In [ ]:
results_df = pd.DataFrame()
partitions_overall = [] 

review_pmids = get_review_pmids()
n_reviews = len(review_pmids)

In [ ]:
from tqdm.auto import tqdm
ground_truth_clusters_df = pd.DataFrame(columns=['Pmid', 'Level', 'Clusters'], dtype=object)
logger.info('Computing ground truth clustering features')
for pmid in tqdm(review_pmids):
    clustering = load_clustering(pmid)
    analyzer = load_analyzer(pmid)
    
    # Pre-calculate all hierarchy levels before grid search to avoid re-calculation of clusterings
    for level in range(1, get_clustering_level(clustering)):
        clusters = preprocess_clustering(
            clustering, level, include_box_sections=False, uniqueness_method='unique_only'
        )
        ground_truth_clusters_df.loc[len(ground_truth_clusters_df)] = (pmid, level, len(set(clusters.values())))
display(ground_truth_clusters_df.head())

In [ ]:
sns.histplot(data=ground_truth_clusters_df, x='Clusters', hue='Level', element='poly')
plt.title('Ground truth clusters number')
plt.savefig(f'figures/{OUTPUT_NAME}_ground_truth_clusters.pdf')
plt.show()

## Grid search
See `grid_search.py` file to launch parameters grid search in parallel with Celery.

In [ ]:
metrics = [adjusted_mutual_info_score, pd_score, reg_v_score]

## Visualization

#### Load results if needed

In [ ]:
load_from_disk = False

In [ ]:
if load_from_disk:
    results_df = pd.read_csv(f'{OUTPUT_NAME}.csv')

#### Extract parameter columns

In [ ]:
score_columns = set([m.__name__ for m in metrics])
param_columns = list(set(results_df.columns) - score_columns - set(['level', 'n_clusters', 'pmid']))
print(param_columns)

#### Number of clusters and adjusted mutual information

In [ ]:
sns.boxplot(x='method', y='n_clusters', hue='method', data=results_df)
plt.title('Clusters number')
plt.xlabel('Method')
plt.ylabel('Clusters')
plt.savefig(f'figures/{OUTPUT_NAME}_clusters_number.pdf')
plt.show()

In [ ]:
sns.boxplot(x='method', y='adjusted_mutual_info_score', hue='level', data=results_df)
plt.title('Adjusted mutual information')
plt.xlabel('Method')
plt.ylabel('AMI')
plt.savefig(f'figures/{OUTPUT_NAME}_adjusted_mutual_information.pdf')
plt.show()

#### Average Scores 

In [ ]:
def get_top_parameter_sets_for_method(score_df, param_cols, method, target_col, n=5):
    return score_df[score_df.method == method].groupby(param_cols)[[target_col, 'n_clusters']].mean().sort_values(by=target_col, 
                                                                                                                  ascending=False).head(n).reset_index()

In [ ]:
def get_top_mean_score_for_method(score_df, param_cols, method, target_col):
    return score_df[score_df.method == method].groupby(param_cols)[target_col].mean().sort_values(ascending=False).values[0]

In [ ]:
target_col = 'adjusted_mutual_info_score'

for method in results_df.method.unique():
    top_score = get_top_mean_score_for_method(results_df, param_columns, method, target_col)
    print(method, ':', target_col, top_score, '\n')
    print(get_top_parameter_sets_for_method(results_df, param_columns, method, target_col), '\n')

In [ ]:
mean_score_data = []
for method in results_df.method.unique():
    method_data = []
    for metric in metrics:
        top_score = get_top_mean_score_for_method(results_df, param_columns, method, metric.__name__)
        method_data.append(top_score)
    mean_score_data.append((method, *method_data))

In [ ]:
metric_names = [m.__name__ for m in metrics]
mean_score_df = pd.DataFrame(mean_score_data, columns=['method', *metric_names])
mean_score_df.head()

In [ ]:
mean_score_df.to_csv(f'{OUTPUT_NAME}_mean_scores_per_method.csv', index=False)

In [ ]:
p = mean_score_df.plot.bar(x='method', y=metric_names)
fig = p.get_figure()
fig.savefig(f'{OUTPUT_NAME}_mean_scores_per_method.png')

#### Best parameters visualization

In [ ]:
import plotly.graph_objects as go

categories = ['similarity_bibliographic_coupling',
              'similarity_cocitation',
              'similarity_citation',
              'similarity_text']

fig = go.Figure()
for method in results_df.method.unique():
    t = get_top_parameter_sets_for_method(results_df, param_columns, method, target_col)
    r = (t['similarity_bibliographic_coupling'].values[0],
         t['similarity_cocitation'].values[0],
         t['similarity_citation'].values[0],
         t['similarity_text'].values[0])
    if method !='lda':
        fig.add_trace(go.Scatterpolar(
            r=r,
            theta=categories,
            fill='toself',
            name=method
        ))
fig.update_layout(
  polar=dict(
    radialaxis=dict(
      visible=True,
      range=[0, 10]
    )),
  showlegend=False
)
fig.write_image(f'figures/{OUTPUT_NAME}_params.pdf')
fig.show()

#### Average Scores for Different Clustering Levels

In [ ]:
def get_top_parameter_sets_for_level_and_method(score_df, param_cols, level, method, target_col, n=5):
    return score_df[(score_df.method == method) & (score_df.level == level)]\
        .groupby(param_cols)[[target_col, 'n_clusters']].mean().sort_values(by=target_col, 
                                                                            ascending=False).head(n).reset_index()

In [ ]:
def get_top_mean_score_for_level_and_method(score_df, param_cols, level, method, target_col):
    return score_df[(score_df.method == method) & (score_df.level == level)]\
        .groupby(param_cols)[target_col].mean().sort_values(ascending=False).values[0]

In [ ]:
target_col = 'adjusted_mutual_info_score'

for level in results_df.level.unique():
    print(f'LEVEL {level}')
    for method in results_df.method.unique():
        top_score = get_top_mean_score_for_level_and_method(results_df, param_columns, level, method, target_col)
        print(method, ':', target_col, top_score, '\n')
        print(get_top_parameter_sets_for_level_and_method(results_df, param_columns, level, method, target_col), '\n')

In [ ]:
level_mean_score_data = []

for level in results_df.level.unique():
    for method in results_df.method.unique():
        method_data = []
        for metric in metrics:
            top_score = get_top_mean_score_for_level_and_method(results_df, param_columns, level, method, metric.__name__)
            method_data.append(top_score)
        level_mean_score_data.append((level, method, *method_data))

In [ ]:
metric_names = [m.__name__ for m in metrics]
level_mean_score_df = pd.DataFrame(level_mean_score_data, columns=['level', 'method', *metric_names])
level_mean_score_df

In [ ]:
level_mean_score_df.to_csv(f'{OUTPUT_NAME}_mean_scores_per_method_and_level.csv', index=False)

In [ ]:
for level in level_mean_score_df.level.unique():
    p = level_mean_score_df[level_mean_score_df.level == level].plot.bar(x='method', y=metric_names, title=f'Level {level}')
    fig = p.get_figure()
    fig.savefig(f'{OUTPUT_NAME}_mean_scores_per_method_level_{level}.png')

In [ ]:
import plotly.graph_objects as go

categories = ['similarity_bibliographic_coupling',
              'similarity_cocitation',
              'similarity_citation',
              'similarity_text']


for level in results_df.level.unique():
    fig = go.Figure()
    print(f'LEVEL {level}')
    for method in results_df.method.unique():
        t = get_top_parameter_sets_for_level_and_method(results_df, param_columns, level, method, target_col)
        r = (t['similarity_bibliographic_coupling'].values[0],
             t['similarity_cocitation'].values[0],
             t['similarity_citation'].values[0],
             t['similarity_text'].values[0])
        if method !='lda':
            fig.add_trace(go.Scatterpolar(
                r=r,
                theta=categories,
                fill='toself',
                name=method
            ))
    fig.update_layout(
      polar=dict(
        radialaxis=dict(
          visible=True,
          range=[0, 10]
        )),
      showlegend=False
    )
    fig.write_image(f'figures/{OUTPUT_NAME}_params_{level}.pdf')
    fig.show()

In [ ]:
print('Visualization - Done')